In [1]:
import os
import cv2
import torch
import numpy as np
import sys

from groundingdino.util import box_ops
from groundingdino.models import build_model
import groundingdino.datasets.transforms as T
from groundingdino.util.slconfig import SLConfig
from groundingdino.util.utils import clean_state_dict
from groundingdino.util.inference import predict as dino_predict
from groundingdino.util.inference import annotate, load_image
from huggingface_hub import hf_hub_download

from segment_anything import sam_model_registry, SamPredictor

from PIL import Image, ImageOps


/home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
Name1="Frame_00001.tif" #Change name for PIV image used to generate mask
InputPath=os.path.join(os.getcwd(),"input")
GenPath=os.path.join(os.getcwd(),"input",Name1)

In [3]:
repo_id="ShilongLiu/GroundingDINO"
filename="groundingdino_swinb_cogcoor.pth"
config_filename="GroundingDINO_SwinB.cfg.py"
device='cuda'

cache_config_file = hf_hub_download(repo_id=repo_id, filename=config_filename)

args = SLConfig.fromfile(cache_config_file)
Dmodel = build_model(args)
args.device = device

cache_file = hf_hub_download(repo_id=repo_id, filename=filename)
checkpoint = torch.load(cache_file, map_location=device)
log = Dmodel.load_state_dict(clean_state_dict(checkpoint['model']), strict=False)

print("Check the latest .pth from: https://github.com/IDEA-Research/GroundingDINO/releases")

Dmodel.eval()

/home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4382.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased
Check the latest .pth from: https://github.com/IDEA-Research/GroundingDINO/releases


GroundingDINO(
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x DeformableTransformerEncoderLayer(
          (self_attn): MultiScaleDeformableAttention(
            (sampling_offsets): Linear(in_features=256, out_features=256, bias=True)
            (attention_weights): Linear(in_features=256, out_features=128, bias=True)
            (value_proj): Linear(in_features=256, out_features=256, bias=True)
            (output_proj): Linear(in_features=256, out_features=256, bias=True)
          )
          (dropout1): Dropout(p=0.0, inplace=False)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
          (linear1): Linear(in_features=256, out_features=2048, bias=True)
          (dropout2): Dropout(p=0.0, inplace=False)
          (linear2): Linear(in_features=2048, out_features=256, bias=True)
          (dropout3): Dropout(p=0.0, inplace=False)
          (norm2): LayerNorm((256,), eps=1e-05, elem

In [4]:
# Image transformation
image_source, image = load_image(GenPath)
MainImage=Image.open(GenPath).convert('RGB')

InputImage = ImageOps.expand(MainImage,border=300,fill='white')

height, width, depth = np.array(MainImage).shape

boxes, logits, phrases = dino_predict(model=Dmodel,
                                      image=image,
                                      caption="Airfoil",
                                      box_threshold=0.3,
                                      text_threshold=0.3,
                                      device='cuda')
print(f"Height: {height}")
print(f"Width: {width}")

NumpyBox = boxes.squeeze().cpu().numpy()

XCord = int(width*NumpyBox[0])
YCord = int(height*NumpyBox[1])

print(f"X: {XCord}") 
print(f"Y: {YCord}")      
torch.cuda.empty_cache()

# annotated_frame = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
# cv2.namedWindow('Window', cv2.WINDOW_NORMAL)
# # Set initial size
# cv2.resizeWindow('Window', 800, 600)
# cv2.imshow('Window', annotated_frame)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
# cv2.imwrite("/home/leo/MIE498Naturized/FullCycle/Thesis/Figures/GDOutSpace.png", annotated_frame)

/home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/transformers/modeling_utils.py:1052: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1298: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/torch/utils/checkpoint.py:238: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Height: 2052
Width: 4603
X: 2300
Y: 1716


/home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/groundingdino/models/GroundingDINO/transformer.py:862: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


In [5]:

sam_checkpoint = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
model_type = "vit_h"
device = "cuda"

checkpoint = sam_checkpoint
sam_model = sam_model_registry[model_type]()
state_dict = torch.hub.load_state_dict_from_url(checkpoint)
sam_model.load_state_dict(state_dict, strict=True)

sam_model.to(device=device)

predictor=SamPredictor(sam_model)

predictor.set_image(np.array(InputImage))

masks, _, _ = predictor.predict(
    point_coords=np.array([[XCord+300, YCord+300]]), 
    point_labels=np.array([1]),
    multimask_output=False
)

torch.cuda.empty_cache()




In [6]:
print(masks[0].shape)

mask_uint8 = masks[0].astype(np.uint8) * 255
print(mask_uint8.shape)

CroppedMask=mask_uint8[300:height+300, 300:width+300]
print(CroppedMask.shape)

maskimage=Image.fromarray(CroppedMask)
maskimage.show()

(2652, 5203)
(2652, 5203)
(2052, 4603)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [7]:

Source = cv2.imread(GenPath)

color_mask = cv2.cvtColor(CroppedMask, cv2.COLOR_GRAY2BGR)

color_mask[CroppedMask == 255] = [255, 0, 0] # Blue BGR

img = cv2.addWeighted(color_mask, 0.3, Source, 1, 0)

cv2.namedWindow("2w", cv2.WINDOW_NORMAL)
cv2.resizeWindow("2w", 1000, 700)

cv2.imshow('2w', img)
cv2.waitKey(0)
cv2.destroyAllWindows()



QFontDatabase: Cannot find font directory /home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/leo/miniforge3/envs/FlowSegClean/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDat